# ST 554 Homework 9
*By: Cass Crews*

## Introduction

The goal of this notebook is to practice building `MLlib` data pipelines. To do so, we will build pipelines for three separate model classes to tune the models on a training set and then compare the best models from each class using a test set. 

The [dataset](https://archive-beta.ics.uci.edu/dataset/73/mushroom) we will use provides mushroom characteristics for 8124 individual mushrooms. This dataset is available in the UCI Machine Learning Repository. The response variable will be an indicator of whether the mushroom is poisonous or edible. 

## Reading in Modules and Splitting Data

Before we begin working with the dataset, we need to read in the modules and sub-modules we'll need. We'll also need to initiate a `spark` instance. 

In [11]:
# Reading in modules and sub-modules, also initiating spark sequence
import pandas as pd
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

from pyspark.ml.feature import SQLTransformer, StringIndexer, Binarizer, VectorAssembler
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import BinaryClassificationEvaluator

We are now ready to read in the data. Unfortunately, the UCI Repository server is down, so we'll need to read the data in from the local environment. 

As `pandas` provides simpler read-in capabilities, we'll read the dataset in as a `pandas` dataframe. We'll print the first few rows to ensure things read in correctly. 

In [13]:
mushroom_data = pd.read_csv("agaricus-lepiota.data", header = None, names = ["edible", "cap_shape", 
                                                                              "cap_surface", "cap_color", 
                                                                              "bruises", "odor", "gill_attach", 
                                                                              "gill_spacing", "gill_size", 
                                                                              "gill_color", "stalk_shape", "stalk_root", 
                                                                              "stalk_surface_ar", "stalk_surface_br", 
                                                                              "stalk_color_ar", "stalk_color_br", 
                                                                              "veil_type", "veil_color", "ring_number", 
                                                                              "ring_type", "spore_color", 
                                                                              "population", "habitat"])

mushroom_data.head()

,edible,cap_shape,cap_surface,cap_color,bruises,odor,gill_attach,gill_spacing,gill_size,gill_color,...,stalk_surface_br,stalk_color_ar,stalk_color_br,veil_type,veil_color,ring_number,ring_type,spore_color,population,habitat
0,p,x,s,n,t,p,f,c,n,k,...,s,w,w,p,w,o,p,k,s,u
1,e,x,s,y,t,a,f,c,b,k,...,s,w,w,p,w,o,p,n,n,g
2,e,b,s,w,t,l,f,c,b,n,...,s,w,w,p,w,o,p,n,n,m
3,p,x,y,w,t,p,f,c,n,n,...,s,w,w,p,w,o,p,k,s,u
4,e,x,s,g,f,n,f,w,b,k,...,s,w,w,p,w,o,e,n,a,g


That's a lot of categorical variables! In fact, all the variables are categorical variables:

In [14]:
# Capturing data types and non-null counts
mushroom_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8124 entries, 0 to 8123
Data columns (total 23 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   edible            8124 non-null   object
 1   cap_shape         8124 non-null   object
 2   cap_surface       8124 non-null   object
 3   cap_color         8124 non-null   object
 4   bruises           8124 non-null   object
 5   odor              8124 non-null   object
 6   gill_attach       8124 non-null   object
 7   gill_spacing      8124 non-null   object
 8   gill_size         8124 non-null   object
 9   gill_color        8124 non-null   object
 10  stalk_shape       8124 non-null   object
 11  stalk_root        8124 non-null   object
 12  stalk_surface_ar  8124 non-null   object
 13  stalk_surface_br  8124 non-null   object
 14  stalk_color_ar    8124 non-null   object
 15  stalk_color_br    8124 non-null   object
 16  veil_type         8124 non-null   object
 17  veil_color    

The `object` type for each column indicates it is a categorical variable. Thus, the entire dataset is comprised of categorical variables we will need to convert to indicator variables for any non-tree-based models. 

Overall, it does seem the dataset was read in correctly, so let's store it in a `pyspark` SQL dataframe to utilize the `MLlib` functionality. We'll print out the first few rows and columns of this new dataframe to ensure the translation worked. 

In [6]:
mushroom_data_2 = spark.createDataFrame(mushroom_data)
mushroom_data_2.select("edible", "cap_shape", "cap_surface", 
                       "cap_color", "bruises", "odor", "gill_attach").show(5)

+------+---------+-----------+---------+-------+----+-----------+
|edible|cap_shape|cap_surface|cap_color|bruises|odor|gill_attach|
+------+---------+-----------+---------+-------+----+-----------+
|     p|        x|          s|        n|      t|   p|          f|
|     e|        x|          s|        y|      t|   a|          f|
|     e|        b|          s|        w|      t|   l|          f|
|     p|        x|          y|        w|      t|   p|          f|
|     e|        x|          s|        g|      f|   n|          f|
+------+---------+-----------+---------+-------+----+-----------+
only showing top 5 rows


This looks good. 

We are now ready to split the data into training and test sets using the `randomSplit()` method. We'll set aside 20\% of observations for the test set and confirm that actually occurs by printing the observation counts in the two dataframes. 

In [12]:
# Capturing training and test sets
train, test = mushroom_data_2.randomSplit([0.8,0.2], seed = 5)

# Printing observation counts for two dataframes
print(train.count(), test.count())

6489 1635


We have successfully split the full dataset into training and test sets! 

## Discussing Model Metric and Model Classes

### The Model Metric

We need to specify a model metric we will use to evaluate each model when tuning the model classes as well as to evaluate the best model of each class when predicting on the test set. As this is a classification problem, a natural choice is log-loss. It provides a more thorough evaluation of model performance than a more coarse metric such as accuracy. To understand this, let's consider the form of log-loss. In particular, let's consider mushroom edibility to be a binary random variable, $y$, with a value of 1 when a mushroom is edible and a value of 0 when it is not. A candidate model's log-loss for a given observation, $y_i$, and corresponding prediction, $p_i$, is, 

$$\text{log-loss} = -(y_i ln(\hat{p}_i) + (1-y_i) ln(1 - \hat{p}_i)).$$

Note that this metric will be close to 0 when our model correctly predicts mushroom edibility with confidence. Meanwhile, the metric will be larger if the model correctly classifies the obervation with low confidence or incorrectly classifies the observation. Thus, it accounts for both prediction accuracy and prediction confidence. When evaluating a model across multiple unseen observations, we will average the individual log-loss values. We will then select the candidate model that minimizes this average log-loss, which we will simply refer to as log-loss from this point forward. 

### The Model Classes

We now need to determine the model classes we will consider. As this dataset includes only categorical variables, tree-based models are a natural choice: they seamlessly handle indicators and allow for complex relationships among them. Thus, we will consider the random forest model. This model constructs a "forest" of classification trees that are used to predict the response class using majority rule. You may be wondering how we construct 500 or 1,000 trees from a single training set. We do so by repeatedly taking bootstrap samples from the training set and using each sample to fit a single tree. For this model class, we often tune three different hyperparameters: 

- The depth of each tree, or the maximum number of splits allowed along a single branch
- The minimum "leaf" size; that is, the minimum number of training observations allowed in any terminal node of a tree
- The size of the random sample of features considered when creating a new split in a tree.

The last hyperparameter is a key aspect of the random forest, as it ensures that not every tree's first, and often most important, splits are made using the same dominant predictors. This ensures we are not effectively fitting the same tree across each bootstrap sample, which would be a pointless endeavor. 

We will also consider two regularized logistic regression model classes: the logistic regression model with L1 penalty and the logistic regression model with L2 penalty. Both models penalize a standard logistic regression model for the magnitude of the estimated coefficients to prevent the model from becoming too complex given a large number of predictors. They differ in the exact form of that penalty: the L1 penalty penalizes the sum of the absolute value of the coefficients, while the L2 penalty penalizes the sum of the squares of the coefficients. The former results in some variables' coefficients being driven exactly to 0, effectively performing variable selection. Meanwhile, the latter does not perform variable selection but does provide the benefit that it will effectively construct a weighted average effect of any highly colinear predictors, preserving the information in each rather than retaining only one colinear predictor. For both models, we will tune a hyperparameter that determines the degree of penalization. We'll call this hyperparameter $\alpha$ in both cases. 

As a reminder, the logistic regression model is a generalized linear model where the log-odds of the response variable being equal to 1 is linear in the model parameters rather than the response variable itself having a lienar relationship with the predictors. This ensures all resultant model probability predictions are bounded between 0 and 1. 